### Empezamos con fifa_world_ranking

In [1]:
# ==========================================================
# Verificaremos que los tres archivos tengan la misma estructura
# para poder unirlos y analizarlos como un solo dataset.
# ==========================================================

import pandas as pd

# Archivos
files = {
    "2023-07-20": "../data/raw/fifa_world_ranking/fifa_ranking-2023-07-20.csv",
    "2024-04-04": "../data/raw/fifa_world_ranking/fifa_ranking-2024-04-04.csv",
    "2024-06-20": "../data/raw/fifa_world_ranking/fifa_ranking-2024-06-20.csv"
}

dfs = {name: pd.read_csv(path) for name, path in files.items()}

base_cols = None

for name, df in dfs.items():
    print("=" * 70)
    print(f"Archivo: {name}")
    print(f"Dimensiones: {df.shape}")
    print("Columnas:")
    print(df.columns.tolist())
    print("\nTipos de datos:")
    print(df.dtypes)

    if base_cols is None:
        base_cols = df.columns.tolist()
    else:
        print("\n¿Tiene la misma estructura que el primero?:", base_cols == df.columns.tolist())

Archivo: 2023-07-20
Dimensiones: (64757, 8)
Columnas:
['rank', 'country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date']

Tipos de datos:
rank                 int64
country_full           str
country_abrv           str
total_points       float64
previous_points    float64
rank_change          int64
confederation          str
rank_date              str
dtype: object
Archivo: 2024-04-04
Dimensiones: (67261, 8)
Columnas:
['rank', 'country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date']

Tipos de datos:
rank               float64
country_full           str
country_abrv           str
total_points       float64
previous_points    float64
rank_change          int64
confederation          str
rank_date              str
dtype: object

¿Tiene la misma estructura que el primero?: True
Archivo: 2024-06-20
Dimensiones: (67472, 8)
Columnas:
['rank', 'country_full', 'country_abrv', 'total_poin

In [2]:
# ==========================================================
# Uniremos los tres archivos en un solo dataset y revisaremos
# su estructura general antes de iniciar la limpieza.
# ==========================================================

import pandas as pd

# Cargar archivos
files = [
    "../data/raw/fifa_world_ranking/fifa_ranking-2023-07-20.csv",
    "../data/raw/fifa_world_ranking/fifa_ranking-2024-04-04.csv",
    "../data/raw/fifa_world_ranking/fifa_ranking-2024-06-20.csv"
]

# Unir datasets
df = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)

# Información general
print(f"Dimensiones: {df.shape}")

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())

print("\nFilas duplicadas:", df.duplicated().sum())

print("\nNúmero de fechas distintas:", df["rank_date"].nunique())
print("Rango de fechas:", df["rank_date"].min(), "->", df["rank_date"].max())

Dimensiones: (199490, 8)

Tipos de datos:
rank               float64
country_full           str
country_abrv           str
total_points       float64
previous_points    float64
rank_change          int64
confederation          str
rank_date              str
dtype: object

Valores nulos:
rank               17
country_full        0
country_abrv        0
total_points        0
previous_points     0
rank_change         0
confederation       0
rank_date           0
dtype: int64

Filas duplicadas: 129168

Número de fechas distintas: 333
Rango de fechas: 1992-12-31 -> 2024-06-20


In [3]:
# ==========================================================
# Analizaremos los registros con valores nulos en la columna
# 'rank' para decidir si se imputan o se eliminan.
# ==========================================================

# Filas con rank nulo
null_rank = df[df["rank"].isna()]

print(f"Cantidad de filas con rank nulo: {len(null_rank)}")
display(null_rank)

Cantidad de filas con rank nulo: 17


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
131094,NaN,Samoa,SAM,894.26,0.0,0,OFC,2023-10-26
131095,NaN,Eritrea,ERI,855.56,0.0,0,CAF,2023-10-26
131096,NaN,Tonga,TGA,861.81,0.0,0,OFC,2023-10-26
131171,NaN,American Samoa,ASA,900.27,0.0,0,OFC,2023-10-26
131277,NaN,Eritrea,ERI,855.56,0.0,0,CAF,2023-11-30
131385,NaN,Eritrea,ERI,855.56,0.0,0,CAF,2023-12-21
131806,NaN,Eritrea,ERI,855.56,0.0,0,CAF,2024-02-15
131942,NaN,Eritrea,ERI,855.56,0.0,0,CAF,2024-04-04
198357,NaN,Eritrea,ERI,855.56,0.0,0,CAF,2023-10-26
198358,NaN,Tonga,TGA,861.81,0.0,0,OFC,2023-10-26


In [4]:
# ==========================================================
# Eliminaremos los registros con 'rank' nulo porque no tienen
# una posición oficial y representan una fracción mínima.
# ==========================================================

# Eliminar filas con rank nulo
df = df.dropna(subset=["rank"]).reset_index(drop=True)

# Convertir rank a entero
df["rank"] = df["rank"].astype(int)

# Verificación
print(df.isnull().sum())
print(f"\nDimensiones actuales: {df.shape}")

rank               0
country_full       0
country_abrv       0
total_points       0
previous_points    0
rank_change        0
confederation      0
rank_date          0
dtype: int64

Dimensiones actuales: (199473, 8)


In [5]:
# ==========================================================
# Analizaremos los registros duplicados para confirmar si son
# copias exactas antes de eliminarlos.
# ==========================================================

# Cantidad de duplicados exactos
duplicados = df[df.duplicated(keep=False)]

print(f"Cantidad de registros duplicados: {duplicados.shape[0]}")
print(f"Cantidad de grupos duplicados: {df.duplicated().sum()}")

# Mostrar algunos ejemplos
display(duplicados.sort_values(list(df.columns)).head(20))

Cantidad de registros duplicados: 196412
Cantidad de grupos duplicados: 129160


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
49806,1,Argentina,ARG,1419.0,1442.0,0,CONMEBOL,2015-10-01
115741,1,Argentina,ARG,1419.0,1442.0,0,CONMEBOL,2015-10-01
182914,1,Argentina,ARG,1419.0,1442.0,0,CONMEBOL,2015-10-01
49494,1,Argentina,ARG,1425.0,1473.0,0,CONMEBOL,2015-08-06
115397,1,Argentina,ARG,1425.0,1473.0,0,CONMEBOL,2015-08-06
182539,1,Argentina,ARG,1425.0,1473.0,0,CONMEBOL,2015-08-06
49728,1,Argentina,ARG,1442.0,1425.0,0,CONMEBOL,2015-09-03
115502,1,Argentina,ARG,1442.0,1425.0,0,CONMEBOL,2015-09-03
182859,1,Argentina,ARG,1442.0,1425.0,0,CONMEBOL,2015-09-03
49259,1,Argentina,ARG,1473.0,1496.0,-2,CONMEBOL,2015-07-09


In [6]:
# ==========================================================
# Eliminaremos los registros duplicados exactos para conservar
# un único historial limpio de rankings FIFA.
# ==========================================================

# Eliminar duplicados
df = df.drop_duplicates().reset_index(drop=True)

# Verificación
print(f"Dimensiones actuales: {df.shape}")
print(f"Duplicados restantes: {df.duplicated().sum()}")

Dimensiones actuales: (70313, 8)
Duplicados restantes: 0


In [7]:
# ==========================================================
# Convertiremos las columnas al tipo de dato adecuado para
# asegurar consistencia durante el análisis y modelado.
# ==========================================================

# Fecha
df["rank_date"] = pd.to_datetime(df["rank_date"])

# Texto
text_cols = ["country_full", "country_abrv", "confederation"]
for col in text_cols:
    df[col] = df[col].astype("string")

# Verificación
print(df.dtypes)

rank                        int64
country_full               string
country_abrv               string
total_points              float64
previous_points           float64
rank_change                 int64
confederation              string
rank_date          datetime64[us]
dtype: object


In [8]:
# ==========================================================
# Revisaremos las variables categóricas para detectar valores
# inconsistentes, categorías inesperadas o errores de escritura.
# ==========================================================

categorical_cols = ["country_full", "country_abrv", "confederation"]

for col in categorical_cols:
    print("=" * 60)
    print(f"{col}")
    print(f"Valores únicos: {df[col].nunique()}")
    display(df[col].value_counts().sort_index())

country_full
Valores únicos: 232


country_full
Afghanistan       231
Albania           334
Algeria           334
American Samoa    278
Andorra           299
                 ... 
Yemen             334
Yugoslavia        103
Zaire              68
Zambia            334
Zimbabwe          334
Name: count, Length: 232, dtype: int64[pyarrow]

country_abrv
Valores únicos: 217


country_abrv
AFG    231
AIA    295
ALB    334
ALG    334
AND    299
      ... 
YEM    334
YUG    103
ZAI     68
ZAM    334
ZIM    334
Name: count, Length: 217, dtype: int64[pyarrow]

confederation
Valores únicos: 6


confederation
AFC         14917
CAF         18407
CONCACAF    12150
CONMEBOL     3340
OFC          3337
UEFA        18162
Name: count, dtype: int64[pyarrow]

In [9]:
# ==========================================================
# Analizaremos las variables numéricas para detectar valores
# atípicos o inconsistentes antes de continuar.
# ==========================================================

display(df.describe().T)

,count,mean,min,25%,50%,75%,max,std
rank,70313.0,102.695348,1.0,52.0,103.0,153.0,211.0,58.739806
total_points,70313.0,456.230088,0.0,84.0,348.0,674.0,2172.0,427.931123
previous_points,70313.0,454.772191,0.0,81.0,346.0,673.0,2187.0,428.195071
rank_change,70313.0,0.359436,-92.0,-1.0,0.0,2.0,207.0,8.912527
rank_date,70313,2008-09-19 05:44:30.726608,1992-12-31 00:00:00,2001-09-19 00:00:00,2008-10-08 00:00:00,2015-08-06 00:00:00,2024-06-20 00:00:00,NaN


In [10]:
# ==========================================================
# Guardaremos el dataset limpio para utilizarlo en las
# siguientes etapas del proyecto.
# ==========================================================

output_path = "../data/processed/fifa_world_ranking_clean.csv"

df.to_csv(output_path, index=False)

print(f"Dataset guardado en: {output_path}")
print(f"Dimensiones finales: {df.shape}")

Dataset guardado en: ../data/processed/fifa_world_ranking_clean.csv
Dimensiones finales: (70313, 8)


## FIFA World Ranking (`fifa_world_ranking_clean.csv`)

Este archivo corresponde al historial limpio del ranking FIFA, obtenido al unir los tres archivos originales (`fifa_ranking-2023-07-20.csv`, `fifa_ranking-2024-04-04.csv` y `fifa_ranking-2024-06-20.csv`).

Durante el proceso de limpieza se realizaron las siguientes acciones:

- Se unieron los tres archivos en un único dataset.
- Se eliminaron 17 registros con valores nulos en la columna `rank`, ya que no tenían una posición oficial asignada.
- Se eliminaron los registros duplicados exactos generados por la unión de los tres archivos.
- Se corrigieron los tipos de datos, convirtiendo `rank_date` al tipo `datetime` y las variables categóricas al tipo `string`.
- Se verificó que no existieran valores inconsistentes en las variables categóricas ni valores numéricos fuera de rango.

El dataset final contiene **70,313 registros** y **8 columnas**.

### Descripción de las columnas

| Columna | Descripción |
|---------|-------------|
| `rank` | Posición de la selección en el ranking FIFA para una fecha determinada. |
| `country_full` | Nombre completo de la selección nacional. |
| `country_abrv` | Abreviatura oficial de la selección. |
| `total_points` | Puntos FIFA obtenidos por la selección en esa fecha. |
| `previous_points` | Puntos obtenidos en la publicación anterior del ranking. |
| `rank_change` | Variación de posiciones respecto al ranking anterior. |
| `confederation` | Confederación a la que pertenece la selección (UEFA, CONMEBOL, AFC, CAF, CONCACAF u OFC). |
| `rank_date` | Fecha de publicación del ranking FIFA. |

### ¿Qué representa cada registro?

Cada fila representa el estado de una selección nacional en una fecha específica del ranking FIFA.

Por ejemplo:

| Fecha | Selección | Ranking | Puntos FIFA |
|-------|-----------|---------|-------------|
| 2024-06-20 | Argentina | 1 | 1860.14 |
| 2024-06-20 | Francia | 2 | 1837.47 |
| 2024-06-20 | Brasil | 4 | 1785.61 |

El mismo país puede aparecer múltiples veces porque el ranking se publica periódicamente, formando una serie temporal desde **1992 hasta 2024**.

### Utilidad para el proyecto

Este dataset proporcionará variables importantes para el modelo predictivo del Mundial 2026, tales como:

- Ranking FIFA de cada selección.
- Puntos FIFA acumulados.
- Evolución del ranking a lo largo del tiempo.
- Cambio de posiciones entre publicaciones.
- Confederación de cada selección.

Estas variables podrán integrarse posteriormente con los resultados históricos, el ranking Elo y las estadísticas de jugadores para construir el dataset final utilizado por los modelos de Machine Learning.

### players.csv

In [11]:
# ==========================================================
# Analizaremos la estructura inicial del dataset de jugadores
# para identificar columnas, tipos de datos y problemas de calidad.
# ==========================================================

import pandas as pd

# Cargar dataset
players = pd.read_csv("../data/raw/players_dataset/players.csv")

# Información general
print("Dimensiones:")
print(players.shape)

print("\nColumnas:")
print(players.columns.tolist())

print("\nTipos de datos:")
print(players.dtypes)

print("\nValores nulos:")
print(players.isnull().sum())

print("\nDuplicados:")
print(players.duplicated().sum())

print("\nPrimeras filas:")
display(players.head())

Dimensiones:
(1248, 72)

Columnas:
['player', 'team', 'team_country', 'position', 'age', 'birth_year', 'club', 'games', 'games_starts', 'minutes', 'minutes_90s', 'goals', 'assists', 'goals_assists', 'goals_pens', 'pens_made', 'pens_att', 'cards_yellow', 'cards_red', 'goals_per90', 'assists_per90', 'goals_assists_per90', 'goals_pens_per90', 'goals_assists_pens_per90', 'shots', 'shots_on_target', 'shots_on_target_pct', 'shots_per90', 'shots_on_target_per90', 'goals_per_shot', 'goals_per_shot_on_target', 'minutes_per_game', 'minutes_pct', 'minutes_per_start', 'games_complete', 'games_subs', 'minutes_per_sub', 'unused_subs', 'points_per_game', 'on_goals_for', 'on_goals_against', 'plus_minus', 'plus_minus_per90', 'plus_minus_wowy', 'cards_yellow_red', 'fouls', 'fouled', 'offsides', 'crosses', 'interceptions', 'tackles_won', 'pens_won', 'pens_conceded', 'own_goals', 'gk_games', 'gk_games_starts', 'gk_minutes', 'gk_goals_against', 'gk_goals_against_per90', 'gk_shots_on_target_against', 'gk_sa

,player,team,team_country,position,age,birth_year,club,games,games_starts,minutes,...,gk_wins,gk_ties,gk_losses,gk_clean_sheets,gk_clean_sheets_pct,gk_pens_att,gk_pens_allowed,gk_pens_saved,gk_pens_missed,gk_pens_save_pct
0,Achref Abada,Algeria,Algeria,DF,27-022,1999,NaN,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Adil Boulbina,Algeria,Algeria,FW,23-066,2003,NaN,2,0,19.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Amine Gouiri,Algeria,Algeria,FW,26-141,2000,Marseille,4,3,251.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Anis Hadj Moussa,Algeria,Algeria,MF,24-146,2002,Feyenoord,3,1,98.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aïssa Mandi,Algeria,Algeria,DF,34-258,1991,Lille,4,4,360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# ==========================================================
# Analizaremos la distribución de jugadores por posición y
# selección para conocer la estructura del dataset.
# ==========================================================

print("Cantidad de jugadores por posición:")
display(players["position"].value_counts())

print("\nCantidad de jugadores por selección:")
display(players["team_country"].value_counts())

Cantidad de jugadores por posición:


position
MF       376
DF       363
FW       171
GK       145
FW,MF     90
DF,MF     51
MF,FW     35
MF,DF     12
DF,FW      5
Name: count, dtype: int64


Cantidad de jugadores por selección:


team_country
Algeria                   26
Argentina                 26
Australia                 26
Austria                   26
Belgium                   26
Bosnia and Herzegovina    26
Brazil                    26
Cape Verde                26
Canada                    26
Colombia                  26
Congo DR                  26
Croatia                   26
Curaçao                   26
Czech Republic            26
Côte d'Ivoire             26
Ecuador                   26
Egypt                     26
England                   26
France                    26
Germany                   26
Ghana                     26
Haiti                     26
Iran                      26
Iraq                      26
Japan                     26
Jordan                    26
Korea Republic            26
Mexico                    26
Morocco                   26
Netherlands               26
New Zealand               26
Norway                    26
Panama                    26
Paraguay                  26
P

In [13]:
# ==========================================================
# Calcularemos el porcentaje de valores nulos por columna para
# identificar variables útiles y variables descartables.
# ==========================================================

null_percentage = (players.isnull().sum() / len(players)) * 100

null_analysis = pd.DataFrame({
    "nulos": players.isnull().sum(),
    "porcentaje": null_percentage
})

display(
    null_analysis.sort_values(
        by="porcentaje",
        ascending=False
    )
)

,nulos,porcentaje
pens_conceded,1248,100.000000
pens_won,1248,100.000000
gk_pens_save_pct,1234,98.878205
gk_save_pct,1191,95.432692
gk_clean_sheets_pct,1191,95.432692
...,...,...
position,0,0.000000
unused_subs,0,0.000000
games_subs,0,0.000000
games_complete,0,0.000000


In [14]:
# ==========================================================
# Calcularemos el porcentaje de valores nulos por columna para
# identificar variables útiles y variables descartables.
# ==========================================================

null_percentage = (players.isnull().sum() / len(players)) * 100

null_analysis = pd.DataFrame({
    "nulos": players.isnull().sum(),
    "porcentaje": null_percentage
})

display(
    null_analysis.sort_values(
        by="porcentaje",
        ascending=False
    )
)

,nulos,porcentaje
pens_conceded,1248,100.000000
pens_won,1248,100.000000
gk_pens_save_pct,1234,98.878205
gk_save_pct,1191,95.432692
gk_clean_sheets_pct,1191,95.432692
...,...,...
position,0,0.000000
unused_subs,0,0.000000
games_subs,0,0.000000
games_complete,0,0.000000


In [15]:
# ==========================================================
# Analizaremos la cantidad de valores únicos por columna para
# detectar variables constantes o con muy poca información.
# ==========================================================

unique_analysis = pd.DataFrame({
    "valores_unicos": players.nunique(dropna=False)
})

display(
    unique_analysis.sort_values(
        by="valores_unicos"
    )
)

,valores_unicos
pens_won,1
pens_conceded,1
cards_yellow_red,2
cards_red,3
gk_pens_saved,3
...,...
plus_minus_per90,402
minutes_pct,480
plus_minus_wowy,545
age,1115


In [16]:
# ==========================================================
# Analizaremos las columnas con un único valor para verificar
# si pueden eliminarse por no aportar información.
# ==========================================================

constant_columns = [
    col for col in players.columns
    if players[col].nunique(dropna=False) == 1
]

print("Columnas constantes:")
print(constant_columns)

for col in constant_columns:
    print(f"\n{col}")
    print(players[col].value_counts(dropna=False))

Columnas constantes:
['pens_won', 'pens_conceded']

pens_won
pens_won
NaN    1248
Name: count, dtype: int64

pens_conceded
pens_conceded
NaN    1248
Name: count, dtype: int64


In [17]:
# ==========================================================
# Eliminaremos las columnas completamente vacías (100% nulas),
# ya que no aportan información al modelo.
# ==========================================================

players = players.drop(
    columns=[
        "pens_won",
        "pens_conceded"
    ]
)

print("Nueva dimensión del dataset:")
print(players.shape)

print("\nColumnas restantes:")
print(len(players.columns))

Nueva dimensión del dataset:
(1248, 70)

Columnas restantes:
70


In [18]:
# ==========================================================
# Analizaremos las columnas con un alto porcentaje de valores
# nulos (>= 90%) para decidir cuáles conservar y cuáles eliminar.
# ==========================================================

null_percentage = (players.isnull().mean() * 100).sort_values(ascending=False)

high_nulls = pd.DataFrame({
    "porcentaje_nulos": null_percentage
})

high_nulls = high_nulls[high_nulls["porcentaje_nulos"] >= 90]

display(high_nulls)

,porcentaje_nulos
gk_pens_save_pct,98.878205
gk_save_pct,95.432692
gk_clean_sheets_pct,95.432692
gk_ties,95.192308
gk_wins,95.192308
gk_shots_on_target_against,95.192308
gk_pens_missed,95.192308
gk_pens_saved,95.192308
gk_games,95.192308
gk_losses,95.192308


In [19]:
# ==========================================================
# Verificaremos si las columnas de porteros contienen datos
# únicamente para jugadores cuya posición es "GK".
# ==========================================================

gk_columns = [col for col in players.columns if col.startswith("gk_")]

print("Distribución de posiciones:")
print(players["position"].value_counts())

print("\nCantidad de porteros:")
gk_players = players[players["position"] == "GK"]
print(len(gk_players))

print("\nValores nulos en columnas de porteros (solo jugadores GK):")
display(
    gk_players[gk_columns].isnull().sum().to_frame("nulos")
)

print("\nDimensión del subconjunto de porteros:")
print(gk_players.shape)

Distribución de posiciones:
position
MF       376
DF       363
FW       171
GK       145
FW,MF     90
DF,MF     51
MF,FW     35
MF,DF     12
DF,FW      5
Name: count, dtype: int64

Cantidad de porteros:
145

Valores nulos en columnas de porteros (solo jugadores GK):


,nulos
gk_games,85
gk_games_starts,85
gk_minutes,85
gk_goals_against,85
gk_goals_against_per90,85
gk_shots_on_target_against,85
gk_saves,85
gk_save_pct,88
gk_wins,85
gk_ties,85



Dimensión del subconjunto de porteros:
(145, 70)


In [20]:
# ==========================================================
# Verificaremos cuántos jugadores son porteros y cuántos de
# ellos realmente tienen estadísticas registradas.
# ==========================================================

gk_players = players[players["position"] == "GK"]

print("Total de porteros:")
print(len(gk_players))

print("\nPorteros con minutos registrados:")
print(gk_players["gk_minutes"].notna().sum())

print("\nPorteros sin minutos registrados:")
print(gk_players["gk_minutes"].isna().sum())

print("\nListado de porteros sin estadísticas:")
display(
    gk_players[gk_players["gk_minutes"].isna()][
        ["player", "team_country", "age", "minutes", "games", "gk_minutes"]
    ]
)

Total de porteros:
145

Porteros con minutos registrados:
60

Porteros sin minutos registrados:
85

Listado de porteros sin estadísticas:


,player,team_country,age,minutes,games,gk_minutes
12,Melvin Mastil,Algeria,26-138,NaN,0,NaN
33,Gerónimo Rulli,Argentina,34-048,NaN,0,NaN
37,Juan Musso,Argentina,32-062,NaN,0,NaN
75,Paul Izzo,Australia,31-182,NaN,0,NaN
86,Florian Wiegele,Austria,25-108,NaN,0,NaN
...,...,...,...,...,...,...
1145,Altay Bayındır,Turkey,28-084,NaN,0,NaN
1157,Mert Günok,Turkey,37-128,NaN,0,NaN
1175,Chris Brady,United States,22-126,NaN,0,NaN
1219,Santiago Mele,Uruguay,28-304,NaN,0,NaN


In [21]:
# ==========================================================
# Analizaremos las columnas relacionadas con minutos y partidos
# para identificar si los valores nulos corresponden únicamente
# a jugadores que no disputaron ningún encuentro.
# ==========================================================

general_columns = [
    "minutes",
    "minutes_90s",
    "games",
    "games_starts",
    "games_complete",
    "games_subs",
    "unused_subs"
]

print("Valores nulos por columna:")
display(players[general_columns].isnull().sum().to_frame("nulos"))

print("\nJugadores sin partidos disputados:")
no_games = players[players["games"] == 0]
print(no_games.shape)

print("\nJugadores con games = 0 y minutos registrados:")
display(
    no_games[
        no_games["minutes"].notna()
    ][
        ["player", "team_country", "games", "minutes"]
    ]
)

print("\nJugadores con games > 0 pero minutos nulos:")
display(
    players[
        (players["games"] > 0) &
        (players["minutes"].isna())
    ][
        ["player", "team_country", "games", "minutes"]
    ]
)

Valores nulos por columna:


,nulos
minutes,221
minutes_90s,221
games,0
games_starts,0
games_complete,0
games_subs,0
unused_subs,0



Jugadores sin partidos disputados:
(221, 70)

Jugadores con games = 0 y minutos registrados:


,player,team_country,games,minutes



Jugadores con games > 0 pero minutos nulos:


,player,team_country,games,minutes


In [22]:
# ==========================================================
# Verificaremos si todas las estadísticas de rendimiento son
# nulas cuando un jugador no disputó ningún partido. Si es así,
# podremos imputarlas posteriormente con 0 de forma segura.
# ==========================================================

exclude_columns = [
    "player",
    "team_country",
    "position",
    "age",
    "birth_year"
]

stat_columns = [
    col for col in players.columns
    if col not in exclude_columns
]

no_games = players[players["games"] == 0]

nulls_no_games = no_games[stat_columns].isnull().sum().sort_values(ascending=False)

display(
    pd.DataFrame({
        "nulos": nulls_no_games,
        "%_nulos": (nulls_no_games / len(no_games) * 100).round(2)
    })
)

,nulos,%_nulos
club,221,100.0
gk_minutes,221,100.0
minutes,221,100.0
goals,221,100.0
minutes_90s,221,100.0
...,...,...
unused_subs,0,0.0
games,0,0.0
games_subs,0,0.0
games_complete,0,0.0


In [23]:
# ==========================================================
# Identificaremos todas las columnas que contienen al menos un
# valor nulo (excepto las de porteros) para preparar la
# imputación de jugadores que no disputaron partidos.
# ==========================================================

gk_columns = [col for col in players.columns if col.startswith("gk_")]

columns_with_nulls = [
    col for col in players.columns
    if players[col].isnull().sum() > 0 and col not in gk_columns
]

null_summary = pd.DataFrame({
    "nulos": players[columns_with_nulls].isnull().sum(),
    "porcentaje": (
        players[columns_with_nulls].isnull().mean() * 100
    ).round(2),
    "dtype": players[columns_with_nulls].dtypes.astype(str)
})

display(
    null_summary.sort_values(
        by="porcentaje",
        ascending=False
    )
)

,nulos,porcentaje,dtype
goals_per_shot_on_target,866,69.39,float64
minutes_per_sub,700,56.09,float64
goals_per_shot,582,46.63,float64
shots_on_target_pct,582,46.63,float64
minutes_per_start,478,38.30,float64
club,353,28.29,str
plus_minus_wowy,344,27.56,float64
minutes,221,17.71,float64
minutes_90s,221,17.71,float64
pens_att,221,17.71,float64


In [24]:
# ==========================================================
# Imputaremos las estadísticas de jugadores que no disputaron
# partidos. Los valores nulos representan ausencia de juego,
# por lo que se reemplazan por 0 en las variables numéricas
# generales (excepto las estadísticas exclusivas de porteros).
# ==========================================================

gk_columns = [col for col in players.columns if col.startswith("gk_")]

numeric_columns = players.select_dtypes(include="number").columns

general_numeric = [
    col for col in numeric_columns
    if col not in gk_columns
]

mask = players["games"] == 0

players.loc[mask, general_numeric] = (
    players.loc[mask, general_numeric]
    .fillna(0)
)

print("Valores nulos restantes:")
display(players.isnull().sum().sort_values(ascending=False).head(30))

Valores nulos restantes:


gk_pens_save_pct              1234
gk_save_pct                   1191
gk_clean_sheets_pct           1191
gk_ties                       1188
gk_wins                       1188
gk_shots_on_target_against    1188
gk_pens_missed                1188
gk_pens_saved                 1188
gk_games                      1188
gk_losses                     1188
gk_games_starts               1188
gk_saves                      1188
gk_clean_sheets               1188
gk_minutes                    1188
gk_pens_att                   1188
gk_pens_allowed               1188
gk_goals_against_per90        1188
gk_goals_against              1188
goals_per_shot_on_target       645
minutes_per_sub                479
shots_on_target_pct            361
goals_per_shot                 361
club                           353
minutes_per_start              257
plus_minus_wowy                123
age                              0
team                             0
player                           0
position            

In [25]:
# ==========================================================
# Analizaremos los jugadores que no tienen club registrado
# para decidir si imputar, conservar o eliminar la columna.
# ==========================================================

club_nulls = players[players["club"].isna()]

print("Jugadores sin club:", len(club_nulls))

display(
    club_nulls[
        ["player", "team_country", "position", "games", "minutes"]
    ].head(30)
)

print("\n¿Todos tienen games = 0?")
print(club_nulls["games"].value_counts())

print("\n¿Todos tienen minutes = 0?")
print(club_nulls["minutes"].value_counts())

Jugadores sin club: 353


,player,team_country,position,games,minutes
0,Achref Abada,Algeria,DF,0,0.0
1,Adil Boulbina,Algeria,FW,2,19.0
12,Melvin Mastil,Algeria,GK,0,0.0
13,Mohamed Amine Tougai,Algeria,DF,0,0.0
17,Oussama Benbot,Algeria,GK,1,90.0
24,Yassine Titraoui,Algeria,MF,0,0.0
25,Zineddine Belaïd,Algeria,DF,2,25.0
33,Gerónimo Rulli,Argentina,GK,0,0.0
37,Juan Musso,Argentina,GK,0,0.0
58,Cammy Devlin,Australia,MF,0,0.0



¿Todos tienen games = 0?
games
0    221
3     43
2     39
1     37
4     12
5      1
Name: count, dtype: int64

¿Todos tienen minutes = 0?
minutes
0.0      221
270.0      9
90.0       6
180.0      6
390.0      4
        ... 
122.0      1
103.0      1
95.0       1
9.0        1
237.0      1
Name: count, Length: 86, dtype: int64


In [26]:
# ==========================================================
# Analizaremos la variable "club" para determinar si los nulos
# corresponden a un patrón específico o si conviene imputarlos
# con una categoría como "Unknown".
# ==========================================================

print("Porcentaje de nulos en 'club':")
print(round(players["club"].isna().mean() * 100, 2), "%")

print("\nCantidad de clubes diferentes:")
print(players["club"].nunique())

print("\nClubes más frecuentes:")
display(players["club"].value_counts().head(20))

print("\nJugadores con partidos disputados y club nulo:")
display(
    players[
        (players["club"].isna()) &
        (players["games"] > 0)
    ][
        ["player", "team_country", "club", "games", "minutes"]
    ].sort_values(["team_country", "player"])
)

Porcentaje de nulos en 'club':
28.29 %

Cantidad de clubes diferentes:
313

Clubes más frecuentes:


club
Manchester City    18
Bayern Munich      17
Arsenal            13
PSG                13
Barcelona          12
Galatasaray        12
Al-Hilal           12
Dortmund           11
Crystal Palace     11
Liverpool          10
Atlético Madrid    10
Real Madrid        10
PSV                10
Aston Villa         9
Milan               9
Lille               8
Tottenham           8
Manchester Utd      8
Benfica             8
Celtic              8
Name: count, dtype: int64


Jugadores con partidos disputados y club nulo:


,player,team_country,club,games,minutes
1,Adil Boulbina,Algeria,NaN,2,19.0
17,Oussama Benbot,Algeria,NaN,1,90.0
25,Zineddine Belaïd,Algeria,NaN,2,25.0
143,Ivan Bašić,Bosnia and Herzegovina,NaN,3,179.0
144,Ivan Šunjić,Bosnia and Herzegovina,NaN,4,196.0
...,...,...,...,...,...
1241,Otabek Shukurov,Uzbekistan,NaN,3,237.0
1242,Ruslanbek Jiyanov,Uzbekistan,NaN,1,1.0
1244,Sherzod Esanov,Uzbekistan,NaN,1,1.0
1245,Sherzod Nasrullaev,Uzbekistan,NaN,3,180.0


In [27]:
# ==========================================================
# Limpiaremos los valores nulos restantes.
#
# - club: se imputa como "Unknown".
# - Variables numéricas (excepto gk_): se reemplazan por 0,
#   ya que los nulos provienen de métricas no aplicables
#   (divisiones por cero o ausencia de eventos).
# - Las variables de porteros (gk_) se conservan tal como están.
# ==========================================================

# Club
players["club"] = players["club"].fillna("Unknown")

# Columnas de porteros
gk_columns = [col for col in players.columns if col.startswith("gk_")]

# Variables numéricas generales
numeric_columns = players.select_dtypes(include="number").columns

general_numeric = [
    col for col in numeric_columns
    if col not in gk_columns
]

players[general_numeric] = players[general_numeric].fillna(0)

print("Nulos restantes:")
display(players.isnull().sum().sort_values(ascending=False).head(25))

Nulos restantes:


gk_pens_save_pct              1234
gk_save_pct                   1191
gk_clean_sheets_pct           1191
gk_ties                       1188
gk_wins                       1188
gk_shots_on_target_against    1188
gk_pens_missed                1188
gk_pens_saved                 1188
gk_games                      1188
gk_losses                     1188
gk_games_starts               1188
gk_saves                      1188
gk_clean_sheets               1188
gk_minutes                    1188
gk_pens_att                   1188
gk_pens_allowed               1188
gk_goals_against_per90        1188
gk_goals_against              1188
team                             0
player                           0
position                         0
team_country                     0
birth_year                       0
age                              0
club                             0
dtype: int64

In [28]:
# ==========================================================
# Imputaremos las estadísticas de porteros con 0.
# Los valores nulos corresponden a jugadores que no son
# porteros o que no disputaron minutos como porteros.
# ==========================================================

gk_columns = [col for col in players.columns if col.startswith("gk_")]

players[gk_columns] = players[gk_columns].fillna(0)

print("Nulos restantes:")
print(players.isnull().sum().sum())

Nulos restantes:
0


In [29]:
# ==========================================================
# Guardaremos el dataset limpio en la carpeta processed.
# ==========================================================

output_path = "../data/processed/players_clean.csv"

players.to_csv(output_path, index=False)

print(f"Dataset guardado en: {output_path}")
print(players.shape)

Dataset guardado en: ../data/processed/players_clean.csv
(1248, 70)


## Players Dataset

Este dataset contiene información de los jugadores convocados por las distintas selecciones nacionales, incluyendo datos demográficos, posición, club y estadísticas individuales de rendimiento.

Durante el preprocesamiento se realizaron las siguientes tareas:

- Se eliminaron las columnas `pens_won` y `pens_conceded` por contener un 100% de valores nulos.
- Se imputaron con `0` las variables numéricas correspondientes a jugadores que no disputaron minutos.
- Se reemplazaron los valores nulos de la columna `club` por `"Unknown"`.
- Se imputaron con `0` las estadísticas específicas de porteros (`gk_*`) cuando no aplicaban o no existían registros.
- Se verificó la ausencia de registros duplicados y se obtuvo un dataset completamente libre de valores nulos.

El resultado es un dataset limpio y consistente, listo para ser integrado con los demás datasets del proyecto y utilizado en la construcción del modelo predictor del Mundial FIFA 2026.

### results_1872_2024

In [31]:
# ==========================================================
# Analizaremos la estructura general del dataset de resultados
# históricos para identificar valores nulos y duplicados.
# ==========================================================

import pandas as pd

results = pd.read_csv("../data/raw/results_1872_2024/results.csv")

print("Dimensiones:", results.shape)
print("\nTipos de datos:")
display(results.dtypes)

print("\nValores nulos:")
display(results.isnull().sum())

print("\nDuplicados:", results.duplicated().sum())

Dimensiones: (49502, 9)

Tipos de datos:


date              str
home_team         str
away_team         str
home_score    float64
away_score    float64
tournament        str
city              str
country           str
neutral          bool
dtype: object


Valores nulos:


date          0
home_team     0
away_team     0
home_score    7
away_score    7
tournament    0
city          0
country       0
neutral       0
dtype: int64


Duplicados: 0


In [32]:
# ==========================================================
# Analizaremos los registros con marcadores faltantes para
# decidir si deben eliminarse del dataset.
# ==========================================================

display(
    results[
        results["home_score"].isna() |
        results["away_score"].isna()
    ]
)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49495,2026-07-05,Brazil,Norway,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49496,2026-07-05,Mexico,England,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
49497,2026-07-06,Portugal,Spain,NaN,NaN,FIFA World Cup,Dallas,United States,True
49498,2026-07-06,United States,Belgium,NaN,NaN,FIFA World Cup,Seattle,United States,False
49499,2026-07-06,Argentina,Egypt,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49500,2026-07-06,Switzerland,Colombia,NaN,NaN,FIFA World Cup,Vancouver,Canada,True
49501,2026-07-09,France,Morocco,NaN,NaN,FIFA World Cup,Foxborough,United States,True


In [33]:
# ==========================================================
# Eliminaremos los partidos sin resultado, ya que corresponden
# a encuentros futuros del Mundial 2026.
# ==========================================================

results = results.dropna(
    subset=["home_score", "away_score"]
)

print("Dimensiones:", results.shape)
print("\nValores nulos:")
display(results.isnull().sum())

Dimensiones: (49495, 9)

Valores nulos:


date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

In [34]:
# ==========================================================
# Guardaremos el dataset limpio de resultados históricos.
# ==========================================================

output_path = "../data/processed/results_clean.csv"

results.to_csv(output_path, index=False)

print(f"Dataset guardado en: {output_path}")
print(results.shape)

Dataset guardado en: ../data/processed/results_clean.csv
(49495, 9)


## Results Dataset

Este dataset contiene el historial de partidos internacionales de selecciones nacionales, incluyendo la fecha, equipos participantes, marcador, torneo y sede del encuentro.

Durante el preprocesamiento se realizaron las siguientes tareas:

- Se verificó la ausencia de registros duplicados.
- Se eliminaron los partidos sin resultado registrado, correspondientes a encuentros futuros del Mundial FIFA 2026.
- Se comprobó que el resto de las variables no presentaran valores nulos.

El resultado es un dataset limpio y consistente con los resultados históricos disponibles, listo para integrarse con los demás datasets del proyecto y utilizarse en el entrenamiento del modelo predictor del Mundial FIFA 2026.

### Seguimos en results_1872_2024

In [35]:
import pandas as pd

goalscorers = pd.read_csv("../data/raw/results_1872_2024/goalscorers.csv")
shootouts = pd.read_csv("../data/raw/results_1872_2024/shootouts.csv")
former_names = pd.read_csv("../data/raw/results_1872_2024/former_names.csv")

print("GOALSCORERS")
print(goalscorers.shape)
display(goalscorers.head())

print("\nSHOOTOUTS")
print(shootouts.shape)
display(shootouts.head())

print("\nFORMER_NAMES")
print(former_names.shape)
display(former_names.head())

GOALSCORERS
(47867, 8)


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,False,False
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,False,False
3,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,75.0,False,False
4,1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2.0,False,False



SHOOTOUTS
(681, 5)


,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN



FORMER_NAMES
(36, 4)


,current,former,start_date,end_date
0,Benin,Dahomey,1959-11-08,1975-11-30
1,Burkina Faso,Upper Volta,1960-04-14,1984-08-04
2,Curaçao,Netherlands Antilles,1957-03-03,2010-10-10
3,Czechoslovakia,Bohemia,1903-04-05,1919-01-01
4,Czechoslovakia,Bohemia and Moravia,1939-01-01,1945-05-01


### Integración del dataset de resultados históricos

La carpeta `results_1872_2024` contiene cuatro archivos relacionados con los partidos internacionales de selecciones nacionales. Sin embargo, no todos poseen el mismo nivel de granularidad, por lo que no es conveniente unirlos directamente.

Se tomó como dataset principal `results.csv`, ya que cada fila representa un partido y constituye la unidad de análisis utilizada para entrenar el modelo de predicción.

Durante el proceso de integración se siguió el siguiente criterio:

- **`results.csv`** se utilizó como base del dataset final.
- **`shootouts.csv`** se integró al dataset principal para incorporar información sobre los partidos definidos mediante tanda de penales.
- **`former_names.csv`** se empleó únicamente para normalizar nombres históricos de selecciones nacionales, garantizando la consistencia de los datos.
- **`goalscorers.csv`** no se integró al dataset final debido a que registra información a nivel de cada gol anotado (una fila por gol), mientras que `results.csv` registra información a nivel de partido (una fila por encuentro). Unir ambos archivos directamente produciría duplicación de registros y alteraría la estructura del dataset.

In [36]:
# ==========================================================
# Integraremos la información de las tandas de penales al
# dataset principal de resultados históricos.
# ==========================================================

# Indicador de definición por penales
shootouts["went_to_shootout"] = 1

# Seleccionar únicamente las columnas necesarias
shootouts_merge = shootouts[
    ["date", "home_team", "away_team", "winner", "went_to_shootout"]
]

# Merge con el dataset principal
results = results.merge(
    shootouts_merge,
    on=["date", "home_team", "away_team"],
    how="left"
)

# Completar partidos sin tanda de penales
results["went_to_shootout"] = results["went_to_shootout"].fillna(0).astype(int)
results["winner"] = results["winner"].fillna("No Shootout")

print(results.shape)
display(results.head())

(49495, 11)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,winner,went_to_shootout
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,No Shootout,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,No Shootout,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,No Shootout,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,No Shootout,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,No Shootout,0


In [37]:
# ==========================================================
# Normalizaremos los nombres históricos de las selecciones
# nacionales utilizando el archivo former_names.csv.
# ==========================================================

former_names["start_date"] = pd.to_datetime(former_names["start_date"])
former_names["end_date"] = pd.to_datetime(former_names["end_date"])

results["date"] = pd.to_datetime(results["date"])

for _, row in former_names.iterrows():
    mask = (
        (results["date"] >= row["start_date"]) &
        (results["date"] <= row["end_date"])
    )

    results.loc[
        mask & (results["home_team"] == row["former"]),
        "home_team"
    ] = row["current"]

    results.loc[
        mask & (results["away_team"] == row["former"]),
        "away_team"
    ] = row["current"]

    results.loc[
        mask & (results["winner"] == row["former"]),
        "winner"
    ] = row["current"]

print("Normalización completada.")
print(results.shape)

display(results.head())

Normalización completada.
(49495, 11)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,winner,went_to_shootout
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,No Shootout,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,No Shootout,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,No Shootout,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,No Shootout,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,No Shootout,0


In [38]:
# ==========================================================
# Convertiremos nuevamente la fecha a formato YYYY-MM-DD y
# guardaremos el dataset histórico limpio.
# ==========================================================

results["date"] = results["date"].dt.strftime("%Y-%m-%d")

output_path = "../data/processed/results_historical_clean.csv"

results.to_csv(output_path, index=False)

print(f"Dataset guardado en: {output_path}")
print("Dimensiones finales:", results.shape)

Dataset guardado en: ../data/processed/results_historical_clean.csv
Dimensiones finales: (49495, 11)


### world_cup_historical_stats_48_teams

In [39]:
# ==========================================================
# Analizaremos la estructura general del dataset de ratings
# ELO para identificar valores nulos y registros duplicados.
# ==========================================================

import pandas as pd

elo = pd.read_csv(
    "../data/raw/world_cup_historical_stats_48_teams/elo_ratings_wc2026.csv"
)

print("Dimensiones:", elo.shape)

print("\nTipos de datos:")
display(elo.dtypes)

print("\nValores nulos:")
display(elo.isnull().sum())

print("\nDuplicados:", elo.duplicated().sum())

display(elo.head())

Dimensiones: (4683, 24)

Tipos de datos:


year                 int64
snapshot_date          str
country                str
rank                 int64
country_code           str
rating               int64
rank_max             int64
rating_max           int64
rank_avg             int64
rating_avg           int64
rank_min             int64
rating_min           int64
matches_total        int64
matches_home         int64
matches_away         int64
matches_neutral      int64
wins                 int64
losses               int64
draws                int64
goals_for            int64
goals_against        int64
confederation          str
is_host              int64
wc2026_exit_round      str
dtype: object


Valores nulos:


year                    0
snapshot_date           0
country                 0
rank                    0
country_code            0
rating                  0
rank_max                0
rating_max              0
rank_avg                0
rating_avg              0
rank_min                0
rating_min              0
matches_total           0
matches_home            0
matches_away            0
matches_neutral         0
wins                    0
losses                  0
draws                   0
goals_for               0
goals_against           0
confederation           0
is_host                 0
wc2026_exit_round    4635
dtype: int64


Duplicados: 0


,year,snapshot_date,country,rank,country_code,rating,rank_max,rating_max,rank_avg,rating_avg,...,matches_away,matches_neutral,wins,losses,draws,goals_for,goals_against,confederation,is_host,wc2026_exit_round
0,1901,1901-12-31,England,1,EN,2013,1,2079,2,1989,...,35,0,46,16,11,262,102,UEFA,0,NaN
1,1901,1901-12-31,Scotland,2,SQ,1973,1,2104,1,2018,...,37,0,53,9,12,277,101,UEFA,0,NaN
2,1902,1902-12-31,Argentina,1,AR,2021,1,2021,1,2021,...,1,0,1,0,0,6,0,CONMEBOL,0,NaN
3,1902,1902-12-31,England,2,EN,1995,1,2079,2,1989,...,38,0,47,16,14,266,105,UEFA,0,NaN
4,1902,1902-12-31,Scotland,3,SQ,1983,1,2104,1,2017,...,40,0,56,9,14,293,106,UEFA,0,NaN


In [40]:
# ==========================================================
# Verificaremos dimensiones, valores nulos y registros
# duplicados del dataset ELO.
# ==========================================================

print("Dimensiones:", elo.shape)

print("\nValores nulos:")
display(elo.isnull().sum())

print("\nDuplicados:", elo.duplicated().sum())

Dimensiones: (4683, 24)

Valores nulos:


year                    0
snapshot_date           0
country                 0
rank                    0
country_code            0
rating                  0
rank_max                0
rating_max              0
rank_avg                0
rating_avg              0
rank_min                0
rating_min              0
matches_total           0
matches_home            0
matches_away            0
matches_neutral         0
wins                    0
losses                  0
draws                   0
goals_for               0
goals_against           0
confederation           0
is_host                 0
wc2026_exit_round    4635
dtype: int64


Duplicados: 0


In [41]:
# ==========================================================
# Completaremos los valores faltantes de la variable
# wc2026_exit_round y guardaremos el dataset limpio.
# ==========================================================

elo["wc2026_exit_round"] = elo["wc2026_exit_round"].fillna("Not Qualified")

output_path = "../data/processed/elo_ratings_wc2026_clean.csv"

elo.to_csv(output_path, index=False)

print(f"Dataset guardado en: {output_path}")
print("Dimensiones finales:", elo.shape)

print("\nNulos restantes:")
print(elo.isnull().sum().sum())

Dataset guardado en: ../data/processed/elo_ratings_wc2026_clean.csv
Dimensiones finales: (4683, 24)

Nulos restantes:
0


## ELO Ratings Dataset

Este dataset contiene las calificaciones ELO y estadísticas históricas de las selecciones nacionales utilizadas como indicadores de rendimiento previo al Mundial FIFA 2026.

Durante el preprocesamiento se realizaron las siguientes tareas:

- Se verificó la ausencia de registros duplicados.
- Se comprobó que todas las variables presentaran valores completos, excepto `wc2026_exit_round`.
- Los valores faltantes de `wc2026_exit_round` se reemplazaron por `"Not Qualified"` para mantener la consistencia del dataset y evitar valores nulos.

El resultado es un dataset limpio y consistente, listo para integrarse con los demás datasets del proyecto y utilizarse en el entrenamiento del modelo predictor del Mundial FIFA 2026.